In [1]:
import sys

%cd src
import microtorch.nn as nn
from microtorch.tensor import Tensor, stack, concat, multinomial
from microtorch.optim import Adam, Optimizer
from microtorch.losses import cross_entropy_with_logits_loss, cross_entropy_loss
from microtorch.utils.trainer import Trainer
from microtorch.utils.loss_dict import LossDict
from microtorch.data.data import default_collate_fn, Dataset, DataLoader
from microtorch.einops import einsum
from microtorch.attention import MultiheadAttentionSlow, MultiheadAttentionFast
%cd ..

import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

/Users/renkehohl/Documents/Python/microtorch/src
/Users/renkehohl/Documents/Python/microtorch


In [2]:
with open('datasets/tiny-shakespeare.txt', 'r') as file:
    text = file.read()

In [3]:
chars = sorted(set(text))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [4]:
stoi = { ch:i for i, ch in enumerate(chars)}
itos = { i:ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s] 
decode = lambda l: ''.join([itos[i] for i in l])

In [5]:
print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [6]:
data = Tensor(encode(text))
data

Tensor([18, 47, 56, ..., 45,  8,  0])

In [7]:
print(data.shape)
print(data[:1000])

(1115394,)
Tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43,
        44, 53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39,
        52, 63,  1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1,
        51, 43,  1, 57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31,
        54, 43, 39, 49,  6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56,
        57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39,
        56, 43,  1, 39, 50, 50,  1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56,
        39, 58, 46, 43, 56,  1, 58, 53,  1, 42, 47, 43,  1, 58, 46, 39, 52,
         1, 58, 53,  1, 44, 39, 51, 47, 57, 46, 12,  0,  0, 13, 50, 50, 10,
         0, 30, 43, 57, 53, 50, 60, 43, 42,  8,  1, 56, 43, 57, 53, 50, 60,
        43, 42,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43,
        52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63, 53, 59,  1, 49, 52, 53,
        61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41, 47, 59, 57,  1, 47,
 

In [8]:
# split data into train and validation sets
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [9]:
block_size = 8
train_data[:block_size+1]

Tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [10]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f'when input is {context} the target: {target}')

when input is Tensor([18]) the target: 47
when input is Tensor([18, 47]) the target: 56
when input is Tensor([18, 47, 56]) the target: 57
when input is Tensor([18, 47, 56, 57]) the target: 58
when input is Tensor([18, 47, 56, 57, 58]) the target: 1
when input is Tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is Tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is Tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [12]:
np.random.seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = np.random.randint(0, len(data) - block_size, (batch_size,))
    x = stack([data[i:i+block_size] for i in ix])
    y = stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

In [13]:
x, y = get_batch('train')

In [14]:
x.shape, y.shape

((4, 8), (4, 8))

## GPT Language Model

In [16]:
dropout = nn.Dropout(0.2)

def scaled_dot_product_attention(q: Tensor, k: Tensor, v: Tensor, attn_mask: Tensor | None = None) -> Tensor:
    scale = q.shape[-1] ** -0.5
    w = einsum(q, k, 'bne,bme->bnm') * scale
    if attn_mask is not None:
        w = w.masked_fill(attn_mask, float('-inf'))  # This could cause problems with gradient calculation
    w = w.softmax(dim=-1)
    w = dropout(w)
    return einsum(w, v, 'bnm,bme->bne')

class AttentionHead(nn.Module):
    def __init__(self, embedding_dim: int, head_size: int = 16):
        super().__init__()
        self.qkv = nn.Linear(embedding_dim, head_size * 3, bias=False)
        # self.to_out = nn.Linear(head_size, embedding_dim)

    def forward(self, x: Tensor) -> Tensor:
        B, N, E = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        attn = scaled_dot_product_attention(
            q, k, v, 
            attn_mask=Tensor.tril(Tensor.ones(N, N)) == 0  # keep for decoder blocks - delete for encoder blocks
        ).leaky_relu(negative_slope=0.01)
        return attn
        return self.to_out(attn)

class AttentionHead(nn.Module):
    def __init__(self, embedding_dim: int, head_size: int = 16):
        self.q = nn.Linear(embedding_dim, head_size, bias=False)
        self.k = nn.Linear(embedding_dim, head_size, bias=False)
        self.v = nn.Linear(embedding_dim, head_size, bias=False)
        # self.to_out = nn.Linear(head_size, embedding_dim)

    def forward(self, x: Tensor) -> Tensor:
        B, N, E = x.shape
        q = self.q(x)
        k = self.k(x)
        v = self.v(x)
        attn = scaled_dot_product_attention(
            q, k, v, 
            attn_mask=Tensor.tril(Tensor.ones(N, N)) == 0  # keep for decoder blocks - delete for encoder blocks
        ).leaky_relu(negative_slope=0.01)
        return attn
        # return self.to_out(attn)

class MultiHeadAttention(nn.Module):
    def __init__(self, n_embed: int, nheads: int = 8, head_size: int = 16):
        super().__init__()
        self.heads = nn.ModuleList([AttentionHead(n_embed, head_size) for _ in range(nheads)])
        self.proj = nn.Sequential(
            nn.Linear(n_embed, n_embed),
            nn.Dropout(0.2)
        )

    def forward(self, x):
        out = concat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return out


class MultiheadAttention(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, dropout: float = 0.0) -> None:
        super().__init__()
        self.d_model = embed_dim
        self.nhead = num_heads
        self.dropout = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()
        self.head_size = embed_dim // num_heads
        self.scale = self.head_size ** -0.5
        self.qkv = nn.Linear(embed_dim, embed_dim * 3, bias=False)
        self.proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, input: Tensor, is_causal: bool = True) -> Tensor:
        B, N, E = input.shape
        q, k, v = self.qkv(input).chunk(3, dim=-1)  # B N E
        q = q.reshape(B, N, self.nhead, self.head_size).transpose(1, 2)  # B H N E
        k = k.reshape(B, N, self.nhead, self.head_size).transpose(1, 2)  # B H N E
        v = v.reshape(B, N, self.nhead, self.head_size).transpose(1, 2)  # B H N E
        weights = q @ k.transpose(-2, -1) * self.scale
        if is_causal:
            weights = weights.masked_fill(Tensor.tril(Tensor.ones(N, N)) == 0, float('-inf'))
        weights = weights.softmax(dim=-1)
        weights = self.dropout(weights)
        out = weights @ v  # B H N E
        out = out.transpose(1, 2).reshape(B, N, E)
        out = self.proj(out.leaky_relu(0.01))
        out = self.dropout(out)
        return out


class FeedForward(nn.Module):
    def __init__(self, n_embed: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.LeakyReLU(),
            nn.Linear(4 * n_embed, n_embed),
            nn.Dropout(0.2)
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)


class Block(nn.Module):
    def __init__(self, n_embed: int, n_head: int):
        super().__init__()
        head_size = n_embed // n_head
        # self.sa = MultiheadAttention(n_embed, n_head, dropout=0.1)
        self.sa = MultiheadAttentionFast(n_embed, n_head, dropout=0.1)
        # self.sa = MultiHeadAttention(n_embed, n_head, head_size)
        self.ffwd = FeedForward(n_embed)
        self.ln1 = nn.LayerNorm1d(n_embed)
        self.ln2 = nn.LayerNorm1d(n_embed)

    def forward(self, x: Tensor) -> Tensor:
        x = x + self.sa(self.ln1(x), is_causal=True)
        x = x + self.ffwd(self.ln2(x))
        return x

In [17]:
class GPT(nn.Module):
    def __init__(self, vocab_size: int, n_embed: int, n_blocks: int = 5, n_heads: int = 4): 
        super().__init__()
        self.token_embedding_table = Tensor.randn(vocab_size, n_embed, requires_grad=True)
        self.position_embedding_table = Tensor.randn(block_size, n_embed, requires_grad=True)
        self.blocks = nn.Sequential(
            *[Block(n_embed, n_heads) for _ in range(n_blocks)],
            nn.LayerNorm1d(n_embed),
        )
        self.lm_head = nn.Linear(n_embed, vocab_size)
        
        self.optimizer = Adam(self.params(), lr=1e-4, weight_decay=0)

    def forward(self, idx: Tensor) -> Tensor:
        B, T = idx.shape
        tok_emb = self.token_embedding_table[idx]  # [B, T, E]
        pos_emb = self.position_embedding_table[Tensor.arange(0, T)]  # [B, T, E]
        x = tok_emb + pos_emb
        x = self.blocks(x)
        logits = self.lm_head(x)
        return logits

    def generate(self, idx: Tensor, max_new_tokens: int):
        for _ in range(max_new_tokens):
            logits = self(idx[:, -block_size:])
            logits = logits[:, -1, :]
            probs = logits.softmax(dim=-1)
            idx_next = multinomial(probs, num_samples=1)
            idx = concat([idx, idx_next], dim=-1)
        return idx

    def train_step(self, idx: Tensor, targets: Tensor) -> float:
        logits = self(idx)
        loss = cross_entropy_with_logits_loss(logits, targets)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        return loss.item()

In [75]:
batch_size = 16
block_size = 32

In [76]:
model = GPT(vocab_size, n_embed=32, n_blocks=4, n_heads=4)

In [77]:
params = list(model.params())

In [19]:
55745 - 54977

768

In [20]:
32 * (32 - 8)

768

In [78]:
sum(np.prod(p.shape) for p in params)

55745

In [79]:
x, y = get_batch('train')

In [80]:
x.shape

(16, 32)

In [81]:
model.train()

GPT()

In [82]:
import sys
from tqdm.notebook import tqdm

In [83]:
model.train_step(*get_batch('train'))

4.462810256554947

## MHA - Einsum Implementation

In [84]:
# einsum
epoch = 0
for _ in range(5):
    epoch += 1
    with tqdm(range(5000), desc=f'Epoch {epoch}', file=sys.stdout) as pbar:
        total_loss = []
        for step in pbar:
            loss = model.train_step(*get_batch('train'))
            total_loss.append(loss)
            pbar.set_postfix(loss=np.mean(total_loss))

Epoch 1:   0%|          | 0/5000 [00:00<?, ?it/s]

Epoch 2:   0%|          | 0/5000 [00:00<?, ?it/s]

Epoch 3:   0%|          | 0/5000 [00:00<?, ?it/s]

Epoch 4:   0%|          | 0/5000 [00:00<?, ?it/s]

Epoch 5:   0%|          | 0/5000 [00:00<?, ?it/s]

In [85]:
# einsum
for _ in range(5):
    epoch += 1
    with tqdm(range(5000), desc=f'Epoch {epoch}', file=sys.stdout) as pbar:
        total_loss = []
        for step in pbar:
            loss = model.train_step(*get_batch('train'))
            total_loss.append(loss)
            pbar.set_postfix(loss=np.mean(total_loss))

Epoch 6:   0%|          | 0/5000 [00:00<?, ?it/s]

Epoch 7:   0%|          | 0/5000 [00:00<?, ?it/s]

Epoch 8:   0%|          | 0/5000 [00:00<?, ?it/s]

Epoch 9:   0%|          | 0/5000 [00:00<?, ?it/s]

Epoch 10:   0%|          | 0/5000 [00:00<?, ?it/s]

## MHA - Fast Implementation

In [70]:
epoch = 0
for _ in range(5):
    epoch += 1
    with tqdm(range(5000), desc=f'Epoch {epoch}', file=sys.stdout) as pbar:
        total_loss = []
        for step in pbar:
            loss = model.train_step(*get_batch('train'))
            total_loss.append(loss)
            pbar.set_postfix(loss=np.mean(total_loss))

Epoch 1:   0%|          | 0/5000 [00:00<?, ?it/s]

Epoch 2:   0%|          | 0/5000 [00:00<?, ?it/s]

Epoch 3:   0%|          | 0/5000 [00:00<?, ?it/s]

Epoch 4:   0%|          | 0/5000 [00:00<?, ?it/s]

Epoch 5:   0%|          | 0/5000 [00:00<?, ?it/s]

In [71]:
for _ in range(5):
    epoch += 1
    with tqdm(range(5000), desc=f'Epoch {epoch}', file=sys.stdout) as pbar:
        total_loss = []
        for step in pbar:
            loss = model.train_step(*get_batch('train'))
            total_loss.append(loss)
            pbar.set_postfix(loss=np.mean(total_loss))

Epoch 6:   0%|          | 0/5000 [00:00<?, ?it/s]

Epoch 7:   0%|          | 0/5000 [00:00<?, ?it/s]

Epoch 8:   0%|          | 0/5000 [00:00<?, ?it/s]

Epoch 9:   0%|          | 0/5000 [00:00<?, ?it/s]

Epoch 10:   0%|          | 0/5000 [00:00<?, ?it/s]

## MHA - Single Attention Heads

In [47]:
epoch = 0
for _ in range(5):
    epoch += 1
    with tqdm(range(5000), desc=f'Epoch {epoch}', file=sys.stdout) as pbar:
        total_loss = []
        for step in pbar:
            loss = model.train_step(*get_batch('train'))
            total_loss.append(loss)
            pbar.set_postfix(loss=np.mean(total_loss))

Epoch 1:   0%|          | 0/5000 [00:00<?, ?it/s]

Epoch 2:   0%|          | 0/5000 [00:00<?, ?it/s]

Epoch 3:   0%|          | 0/5000 [00:00<?, ?it/s]

Epoch 4:   0%|          | 0/5000 [00:00<?, ?it/s]

Epoch 5:   0%|          | 0/5000 [00:00<?, ?it/s]

In [49]:
for _ in range(5):
    epoch += 1
    with tqdm(range(5000), desc=f'Epoch {epoch}', file=sys.stdout) as pbar:
        total_loss = []
        for step in pbar:
            loss = model.train_step(*get_batch('train'))
            total_loss.append(loss)
            pbar.set_postfix(loss=np.mean(total_loss))

Epoch 11:   0%|          | 0/5000 [00:00<?, ?it/s]

Epoch 12:   0%|          | 0/5000 [00:00<?, ?it/s]

Epoch 13:   0%|          | 0/5000 [00:00<?, ?it/s]

Epoch 14:   0%|          | 0/5000 [00:00<?, ?it/s]

Epoch 15:   0%|          | 0/5000 [00:00<?, ?it/s]

In [50]:
model.eval()

GPT()

In [51]:
def generate_stream(self, idx: Tensor, max_new_tokens: int, temperature=1.0):
    for _ in range(max_new_tokens):
        logits = self(idx[:, -block_size:])
        logits = logits[:, -1, :] / temperature
        probs = logits.softmax(dim=-1)
        idx_next = multinomial(probs, num_samples=1)
        print(decode(idx_next[0].tolist()), end='')
        idx = concat([idx, idx_next], dim=-1)
    # return idx

In [52]:
idx = Tensor.zeros(1, 1, dtype=int)
generate_stream(model, idx, 1000, temperature=0.6)

COLARY IUIT:
Hord ve my hor,

Ill the by the mall herere therin as but but me ce the the mer mase so my hay plor and beand thance pot all a ling my thoue afu dein the dithe thaty she of cor. thenve pay witer whoust my weal the thou poor so a fall henter mut and thall fowis of and my wouos hothend and on reals of the by ward the my theels hath the shat be hathe the mat'd the the prand, ier rarut lo foo ar mat ingan sunges anor membe bro morte, hit the tonire the an coth so malls the thee spee
Irille wear thatherer besus the hin for the alll'e, wistle he tough on or thee thing dend thing,
I wel bestat se ale, and parense sithe lot
Bind the and sling shours inenes ollad thad le thene thenty om so the me lemoug of for you in is ause ona haing deathals hame ou knit ther ance now here the as ported if it cone for and carue sand pling po ath bran to or lighers that prom to ither shour for thinge buing to stan my, are the arding to the lomer
Wit an the mouke booh the ade we meme the ater sprea